# 05 — Three engines: analytical / ngspice / LTspice

Nominal, **WC hypercube**, and **uniform MC** on the same `CircuitParams`.

| Engine | What it is | WC | MC |
|--------|------------|----|----|
| Analytical | closed-form $V_{\mathrm{OUT}}$ | 16-corner envelope | `N_MC` Uniform |
| ngspice | behavioral LDO, external divider | same 16 corners (OP) | `N_SPICE_MC` OP |
| LTspice | vendor `LT3010.lib` | 16 grouped-sign corners on ADJ injection | `N_LTSPICE_MC` `mc()` |

LTspice **breaks the ADJ net** and injects `VADJ_ERR` (delta from typical
1.275 V) plus `IADJ_EXTRA` so the macromodel's typical reference is not
double-counted.

Set `RUN_LTSPICE_BATCH = True` in the setup cell to batch-run. If logs
already exist they are parsed either way.

## Setup — editable BOM and run knobs

This cell is shared by every notebook. It puts `P5V_Regulator/` on `sys.path`
and builds **one** `CircuitParams` object. Change a resistor, capacitor, or
tolerance here and re-run: analytical, SPICE, WC, and MC all follow this object.

**Plot callouts:** each figure cell has an `ann = [dict(...), ...]` list.
Edit `text`, `xy` (data coords of the arrow tip), and `offset=(dx, dy)` in
**points** to nudge a label. Use `xytext=(x, y)` instead of `offset` for data
coordinates. `arrow=False` for a plain label. `ann = []` hides all callouts.

Tolerances are **fractions** (0.01 = 1 %). Figures go to `results/figures/`.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def _find_project_root() -> Path:
    start = Path.cwd().resolve()
    named = Path("Sections/Electronics/Analog_and_Digital/LT3010_5V_Supply")
    candidates = [start, *start.parents, start / named]
    if start.name == "notebooks":
        candidates.insert(0, start.parent)
    for cand in candidates:
        if (cand / "p5v_design" / "__init__.py").is_file():
            return cand
    return start

ROOT = _find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from p5v_design.params import CircuitParams, ToleranceSpec, format_eng, LdoParams
from p5v_design import analysis, schematic, plots, stability
from p5v_design import plots_wc_mc as pwc
from p5v_design.ldo_model import dropout_v, ignd_a, tj_c, psrr_db, vadj_typical_vs_temp
from dataclasses import replace

# --- EDITABLE BOM ---
# Datasheet TA01, "5V Supply with Shutdown" (LT3010/LT3010-5 Rev. J page 1).
# Fixed 5 V part: SENSE tied to OUT (R1 = 0). No external divider (R2 open).
# Capacitors are the 1 µF parts drawn on that figure. Load is the labeled 50 mA.
# VIN is printed as 5.4 V to 80 V; 12 V is the single-point bias inside that range.
R1 = 0.0             # SENSE shorted to OUT
R2 = 1.0e12          # open — no bottom resistor
CFF = 0.0            # no feedforward capacitor on the fixed part
COUT = 1.0e-6
CIN = 1.0e-6
ESR_OUT = 10e-3      # ceramic assumption, not printed on TA01
ESR_IN = 10e-3
RLOAD = 100.0        # 5 V / 50 mA
R_TOL = 0.01         # 1 %
C_TOL = 0.10         # 10 %
VIN_NOM = 12.0
VIN_MIN = 5.4
VIN_MAX = 80.0
ILOAD_OP = 50e-3
N_MC = 2000
N_SPICE_MC = 40
N_LTSPICE_MC = 200
SKIP_SPICE_WC_MC = False
RUN_LTSPICE_BATCH = False
SPICE_WORKERS = None
LTSPICE_WORKERS = None

p = CircuitParams(
    r1=R1, r2=R2, cff=CFF, cout=COUT, cin=CIN,
    esr_out=ESR_OUT, esr_in=ESR_IN, r_load=RLOAD,
    r_tol=ToleranceSpec(R_TOL), c_tol=ToleranceSpec(C_TOL),
)
p = replace(p, op=replace(p.op, vin_nom=VIN_NOM, vin_min=VIN_MIN, vin_max=VIN_MAX,
                           iload_op=ILOAD_OP, r_load=RLOAD))

FIG = ROOT / "results" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
print("gain =", p.gain, "  Vout nom =", analysis.vout_dc(p))
print("SPICE_WORKERS =", SPICE_WORKERS, "  N_MC =", N_MC)


gain = 1.0   Vout nom = 5.0
SPICE_WORKERS = None   N_MC = 2000


## Write decks and (optionally) batch-run LTspice

In [2]:
from p5v_design.ltspice_io import (
    find_ltspice, run_ltspice_deck, run_ltspice_decks, parse_meas_log,
    parse_meas_tables, parse_ltspice_raw_tran, meas_frame, meas_values,
)
from p5v_design.worst_case import (
    analytical_worst_case, rss_bounds_many, spice_wc_corners, corners_frame,
)
from p5v_design.monte_carlo import sample_metrics, spice_monte_carlo
from p5v_design.simulate import find_ngspice
import importlib.util
lt_py = ROOT / "LTSpice" / "run_ltspice_batch.py"
mod = None
if lt_py.is_file():
    spec = importlib.util.spec_from_file_location("run_ltspice_batch", lt_py)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
else:
    print("LTspice helper is not in this repository. Vendor LT3010.lib is not redistributed.")
mod.write_all(p, mc_n=N_LTSPICE_MC) if mod is not None else None
print("decks written")

HAVE_LT = False
try:
    print("LTspice", find_ltspice())
    HAVE_LT = True
except FileNotFoundError as e:
    print(e)

HAVE_SPICE = False
try:
    print("ngspice", find_ngspice())
    HAVE_SPICE = True
except FileNotFoundError as e:
    print(e)

op_cir = ROOT/"LTSpice"/"Nominal"/"P5V_Regulator_op.cir"
tran_cir = ROOT/"LTSpice"/"Nominal"/"P5V_Regulator_tran.cir"
wc_cir = ROOT/"LTSpice"/"WC"/"P5V_Regulator_WC.cir"
mc_cir = ROOT/"LTSpice"/"MC"/"P5V_Regulator_MC.cir"
if HAVE_LT and RUN_LTSPICE_BATCH:
    run_ltspice_decks([op_cir, wc_cir, mc_cir], max_workers=LTSPICE_WORKERS)
else:
    print("LTspice batch skipped (RUN_LTSPICE_BATCH=False). Existing logs still parsed.")


LTspice helper is not in this repository. Vendor LT3010.lib is not redistributed.
decks written
LTspice.exe not found. Set LTSPICE_EXE.
ngspice /usr/bin/ngspice
LTspice batch skipped (RUN_LTSPICE_BATCH=False). Existing logs still parsed.


## Load all three engines into frames

In [3]:
wc = analytical_worst_case(p)
rss = rss_bounds_many(p)
cdf = corners_frame(wc)
mc = sample_metrics(p, n=N_MC, seed=42)
mc.save(ROOT/"results"/"mc_analytical")

ng_wc = None
ng_mc = None
if HAVE_SPICE and not SKIP_SPICE_WC_MC:
    ng_rows = spice_wc_corners(p, workdir=ROOT/"results"/"ngspice_wc",
                               max_workers=SPICE_WORKERS, full=True)
    ng_wc = pd.DataFrame(ng_rows)
    ng_mc = spice_monte_carlo(p, n=N_SPICE_MC, seed=42,
                              workdir=ROOT/"results"/"ngspice_mc",
                              max_workers=SPICE_WORKERS)
    display(ng_wc[["corner", "vout_an", "vout_spice"]].head())
    display(ng_mc.summary)

lt_wc = meas_frame(ROOT/"LTSpice"/"WC"/"P5V_Regulator_WC.log") if (ROOT/"LTSpice"/"WC"/"P5V_Regulator_WC.log").is_file() else pd.DataFrame()
lt_mc = meas_frame(ROOT/"LTSpice"/"MC"/"P5V_Regulator_MC.log") if (ROOT/"LTSpice"/"MC"/"P5V_Regulator_MC.log").is_file() else pd.DataFrame()
print("LTspice WC N", len(lt_wc), "MC N", len(lt_mc))

def _band_frame(fr, col="vout"):
    if fr is None or len(fr) == 0 or col not in fr.columns:
        return None
    return pwc.band_from_array(fr[col])

bands_vout = {"analytical": pwc.band_from_wc(wc, "vout")}
if ng_wc is not None:
    bands_vout["ngspice"] = _band_frame(ng_wc, "vout_spice")
if len(lt_wc):
    bands_vout["LTspice"] = _band_frame(lt_wc, "vout")
print("Vout bands", bands_vout)

------------------------------------------------------------------------------


  ngspice WC hypercube


  ngspice  jobs=16  workers=8  requested=auto  cpu=8  env P5V_SPICE_WORKERS=''


  queue: R1lo_R2lo_Valo_Ialo, R1lo_R2lo_Valo_Iahi, R1lo_R2lo_Vahi_Ialo, R1lo_R2lo_Vahi_Iahi, R1lo_R2hi_Valo_Ialo, R1lo_R2hi_Valo_Iahi  +10 more


------------------------------------------------------------------------------


...

  [ngspice MC n=40] done  40 ok / 0 fail / 40 jobs  workers=8  wall=2.06s  cpu=13.65s  speedup=6.62× vs serial


------------------------------------------------------------------------------


,corner,vout_an,vout_spice
0,R1lo_R2lo_Valo_Ialo,4.925,4.924951
1,R1lo_R2lo_Valo_Iahi,4.925,4.924951
2,R1lo_R2lo_Vahi_Ialo,5.075,5.074949
3,R1lo_R2lo_Vahi_Iahi,5.075,5.074949
4,R1lo_R2hi_Valo_Ialo,4.925,4.924951


,mean,std,min,max
vout,4.999849,0.047309,4.931242,5.069935
vout_spice,4.999799,0.047309,4.931193,5.069884
headroom,6.700151,0.047309,6.630065,6.768758


LTspice WC N 0 MC N 0
Vout bands {'analytical': (5.0, 4.925, 5.075), 'ngspice': (4.99995, 4.924951, 5.074949)}


## WC range bars — three engines (comparator dual-device chart)

One horizontal bar per engine, endpoint ticks, min/max labels, 5 V goal.
This is the picture that should match notebook 01 of the hysteresis project.

In [4]:
ann_r = [dict(text="5 V", xy=(5.0, 1.0), offset=(8, 14), arrow=True, color="#9467bd")]
pwc.plot_range_bars(
    {k: v for k, v in bands_vout.items() if v is not None},
    metric="vout", target=5.0, rss=(rss["vout"]["low"], rss["vout"]["high"]),
    annotations=ann_r, path=FIG/"05_wc_range_three.png",
    title=r"WC $V_{\mathrm{OUT}}$  analytical / ngspice / LTspice",
)
plt.show()
pwc.plot_whiskers_engines(
    {k: v for k, v in bands_vout.items() if v is not None},
    metric="vout", target=5.0, path=FIG/"05_wc_whiskers_three.png",
)
plt.show()

# Grid of metrics that exist on all analytical + ngspice
grid = {
    "vout": {"analytical": pwc.band_from_wc(wc, "vout")},
    "headroom": {"analytical": pwc.band_from_wc(wc, "headroom")},
    "p_diss": {"analytical": pwc.band_from_wc(wc, "p_diss")},
    "efficiency": {"analytical": pwc.band_from_wc(wc, "efficiency")},
}
if ng_wc is not None:
    grid["vout"]["ngspice"] = _band_frame(ng_wc, "vout_spice")
if len(lt_wc):
    grid["vout"]["LTspice"] = _band_frame(lt_wc, "vout")
pwc.plot_metric_range_grid(
    grid, rss_by_metric={k: (rss[k]["low"], rss[k]["high"]) for k in rss},
    target_by_metric={"vout": 5.0},
    path=FIG/"05_wc_range_grid.png",
    title="WC range bars by metric and engine",
)
plt.show()

spans = {
    "analytical": {m: wc.span(m) for m in ("vout", "headroom", "p_diss")},
}
if ng_wc is not None:
    spans["ngspice"] = {"vout": float(ng_wc.vout_spice.max() - ng_wc.vout_spice.min()),
                        "headroom": np.nan, "p_diss": np.nan}
if len(lt_wc):
    spans["LTspice"] = {"vout": float(lt_wc.vout.max() - lt_wc.vout.min()),
                        "headroom": np.nan, "p_diss": np.nan}
pwc.plot_span_grouped_bars(spans, metrics=("vout",), path=FIG/"05_wc_span_vout.png",
                           title="WC Vout span by engine", ylabel="span (V)")
plt.show()

cell:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cell:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cell:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cell:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Hypercube scatter + stem: analytical corners vs ngspice / LTspice

In [5]:
spice_sc = None
if ng_wc is not None:
    spice_sc = ng_wc.rename(columns={"vout_spice": "vout"})
pwc.plot_hypercube_scatter(
    cdf, x="headroom", y="vout", spice=spice_sc,
    nominal=(wc.nominal["headroom"], wc.nominal["vout"]), target_y=5.0,
    annotations=[dict(text="nominal", xy=(wc.nominal["headroom"], wc.nominal["vout"]), offset=(12, -14), arrow=True)],
    path=FIG/"05_hypercube_scatter.png",
    title="WC hypercube  analytical + ngspice",
)
plt.show()
pwc.plot_corner_strip(
    cdf, metric="vout", spice=spice_sc, target=5.0,
    path=FIG/"05_corner_strip.png",
)
plt.show()
if ng_wc is not None:
    pwc.plot_an_vs_spice(
        ng_wc["vout_an"].to_numpy(), ng_wc["vout_spice"].to_numpy(),
        engine="ngspice", path=FIG/"05_an_vs_ng_corners.png",
    )
    plt.show()
if len(lt_wc):
    # LTspice WC is a .step sweep, not 1:1 with our corner names — compare as a distribution
    pwc.plot_range_bars(
        {
            "analytical": pwc.band_from_wc(wc, "vout"),
            "LTspice": _band_frame(lt_wc, "vout"),
        },
        metric="vout", target=5.0, path=FIG/"05_range_an_lt.png",
        title="Analytical hypercube vs LTspice 16-corner .step",
    )
    plt.show()
    pwc.plot_wc_rss_band(
        nom=wc.nominal["vout"], wc=(wc.minimum["vout"], wc.maximum["vout"]),
        rss=(rss["vout"]["low"], rss["vout"]["high"]), target=5.0,
        mc=lt_wc["vout"].to_numpy(),
        path=FIG/"05_lt_wc_on_band.png",
        title="LTspice WC corners as a rug on the analytical WC/RSS band",
    )
    plt.show()

cell:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cell:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## MC histograms — three engines, shared bins

In [6]:
frames = {"analytical": mc.frame}
if ng_mc is not None:
    frames["ngspice"] = ng_mc.frame.rename(columns={"vout_spice": "vout"})
if len(lt_mc):
    frames["LTspice"] = lt_mc

pwc.plot_mc_hist_engines(
    frames, metric="vout",
    wc=(wc.minimum["vout"], wc.maximum["vout"]),
    rss=(rss["vout"]["low"], rss["vout"]["high"]),
    nominal=wc.nominal["vout"], target=5.0,
    annotations=[dict(text="5 V", xy=(5.0, 0), offset=(10, 32), arrow=True, color="#9467bd")],
    path=FIG/"05_mc_hist_three.png",
    title="MC $V_{OUT}$  analytical / ngspice / LTspice",
)
plt.show()
pwc.plot_cdf(frames, metric="vout", wc=(wc.minimum["vout"], wc.maximum["vout"]),
             target=5.0, path=FIG/"05_mc_cdf_three.png")
plt.show()
pwc.plot_percentile_bars(frames, metric="vout", path=FIG/"05_mc_p01_p99_three.png")
plt.show()
pwc.plot_mc_cloud(
    {k: v for k, v in frames.items() if "headroom" in v.columns},
    x="headroom", y="vout",
    wc_box=(wc.minimum["headroom"], wc.maximum["headroom"], wc.minimum["vout"], wc.maximum["vout"]),
    target_y=5.0, path=FIG/"05_mc_cloud_three.png",
)
plt.show()
for name, fr in frames.items():
    pwc.plot_mc_vs_run(
        fr, metric="vout", wc=(wc.minimum["vout"], wc.maximum["vout"]),
        target=5.0, engine=name, path=FIG/f"05_mc_vs_run_{name}.png",
    )
    plt.show()

cell:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cell:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cell:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cell:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Span summary: WC box vs MC p01–p99 vs RSS, by engine

In [7]:
span_map = {
    "analytical WC": {"vout": wc.span("vout")},
    "analytical MC p01–p99": {
        "vout": float(mc.frame.vout.quantile(0.99) - mc.frame.vout.quantile(0.01))
    },
    "RSS": {"vout": 2 * rss["vout"]["delta_rss"]},
}
if ng_wc is not None:
    span_map["ngspice WC"] = {"vout": float(ng_wc.vout_spice.max() - ng_wc.vout_spice.min())}
if ng_mc is not None and "vout_spice" in ng_mc.frame.columns:
    q = ng_mc.frame.vout_spice.quantile
    span_map["ngspice MC p01–p99"] = {"vout": float(q(0.99) - q(0.01))}
if len(lt_wc):
    span_map["LTspice WC"] = {"vout": float(lt_wc.vout.max() - lt_wc.vout.min())}
if len(lt_mc):
    q = lt_mc.vout.quantile
    span_map["LTspice MC p01–p99"] = {"vout": float(q(0.99) - q(0.01))}
display(pd.DataFrame(span_map).T.rename(columns={"vout": "Vout span (V)"}))
pwc.plot_span_grouped_bars(span_map, metrics=("vout",), path=FIG/"05_span_all.png",
                           title="Vout span  WC / MC / RSS  × engine", ylabel="span (V)")
plt.show()

,Vout span (V)
analytical WC,0.150000
analytical MC p01–p99,0.146845
RSS,0.150000
ngspice WC,0.149998
ngspice MC p01–p99,0.137624


cell:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Nominal LTspice OP / startup (vendor macromodel)

Typical dynamics live here. WC/MC above used ADJ injection; this block is
the unmodified LT3010.lib at the BOM values.

In [8]:
ann = [dict(text="settled Vout", xy=(8.0, analysis.vout_dc(p)), offset=(10, 12), arrow=True)]
if HAVE_LT and (RUN_LTSPICE_BATCH or op_cir.with_suffix(".log").is_file()):
    if RUN_LTSPICE_BATCH or not op_cir.with_suffix(".log").is_file():
        log = run_ltspice_deck(op_cir)
    else:
        log = op_cir.with_suffix(".log")
    print("OP meas", parse_meas_log(log))
    raw = tran_cir.with_suffix(".raw")
    if RUN_LTSPICE_BATCH or not raw.is_file():
        try:
            run_ltspice_deck(tran_cir)
        except Exception as e:
            print("tran skip", e)
    if raw.is_file():
        tr = parse_ltspice_raw_tran(raw)
        plots.plot_tran(tr.t, tr.vout, vin=tr.vin, lt_t=tr.t, lt_v=tr.vout,
                        annotations=ann, path=FIG/"05_startup.png", title="LTspice startup (typical model)")
        plt.show()
else:
    print("LTspice nominal not run. Set RUN_LTSPICE_BATCH = True.")
print("done three-engine comparison")

LTspice nominal not run. Set RUN_LTSPICE_BATCH = True.
done three-engine comparison


## OP bars and PSRR overlay (HV_monitor notebook 05 style)

Grouped $V_{\mathrm{OUT}}$ at the operate point. PSRR overlay uses the
datasheet G21 curve plus ngspice AC; LTspice PSRR if `P5V_Regulator_psrr`
has been run.

In [9]:
rows = {"analytical": {"vout": analysis.vout_dc(p)}}
if HAVE_SPICE:
    try:
        from p5v_design.simulate import simulate_op
        op_ng = simulate_op(p, workdir=ROOT/"results"/"ngspice_nom"/"op")
        rows["ngspice"] = {"vout": float(op_ng.nodes.get("out", next(iter(op_ng.nodes.values()))))}
    except Exception as e:
        print("ngspice OP skip", e)
lt_op = ROOT/"LTSpice"/"Nominal"/"P5V_Regulator_op.log"
if HAVE_LT and lt_op.is_file():
    try:
        m = parse_meas_log(lt_op)
        if "vout_op" in m:
            rows["LTspice"] = {"vout": float(m["vout_op"])}
    except Exception as e:
        print("LTspice OP skip", e)
plots.plot_op_compare_bars(rows, metrics=("vout",), title="OP Vout  three engines",
                           path=FIG/"05_op_bars.png")
plt.show()
ann = [dict(text="120 Hz", xy=(120, 75), offset=(12, 8), arrow=True)]
if HAVE_SPICE:
    try:
        from p5v_design.simulate import simulate_ac_psrr
        ac = simulate_ac_psrr(p, workdir=ROOT/"results"/"ngspice_nom"/"psrr")
        plots.plot_psrr(p, spice_f=ac.f_hz, spice_db=ac.psrr_db, annotations=ann,
                        path=FIG/"05_psrr_overlay.png")
        f = np.logspace(1, 6, 241)
        ds = psrr_db(f, cout=p.cout)
        sp = np.interp(f, ac.f_hz, ac.psrr_db)
        plots.plot_error_db(f, sp - ds, ylabel="ngspice − G21 (dB)",
                            title="PSRR residual", path=FIG/"05_psrr_error.png")
        plt.show()
    except Exception as e:
        print("PSRR overlay skip", e)
        plots.plot_psrr(p, annotations=ann, path=FIG/"05_psrr_overlay.png")
else:
    plots.plot_psrr(p, annotations=ann, path=FIG/"05_psrr_overlay.png")
plt.show()

cell:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cell:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
